# Interview Prep: 13 - Concurrency Deep Dive

Mastering concurrency is what separates a mid-level Python developer from a senior. This notebook explores **asyncio**, **threading**, and **multiprocessing**, and when to apply each to solve real-world bottlenecks.

---

## 1. Concurrency vs. Parallelism

### **Interview Question**: "What is the difference between concurrency and parallelism, and does Python support both?"

**Answer**:
- **Concurrency**: Handling multiple tasks by interleaving them (like a single waiter serving multiple tables). Python supports this via `threading` and `asyncio`.
- **Parallelism**: Executing multiple tasks at the exact same time on different CPU cores (like having multiple waiters). Python supports this via `multiprocessing`.

---

## 2. Choosing the Right Tool

| Strategy | Best For | Mechanism |
| :--- | :--- | :--- |
| **asyncio** | High-scale I/O (WebSockets, many API calls) | Single-threaded Event Loop |
| **threading** | Standard I/O (Filesystem, simple requests) | OS-level threads (bound by GIL) |
| **multiprocessing** | CPU-bound tasks (Image processing, heavy math) | Separate Python processes (bypasses GIL) |

---

## 3. The `asyncio` Event Loop

### **Interview Question**: "What happens if you run a synchronous `time.sleep(5)` inside an `async def` function?"

**Answer**: It **blocks the entire event loop**. Because `asyncio` is single-threaded and cooperative, no other tasks can run while that thread is sleeping. You must always use `await asyncio.sleep(5)` to yield control back to the loop.

---

## 4. Coding Challenge: Bridging Async and Multiprocessing

**Task**: Run a heavy computation (CPU-bound) without blocking your `asyncio` event loop. We use `run_in_executor` with a `ProcessPoolExecutor`.


In [ ]:
import asyncio
from concurrent.futures import ProcessPoolExecutor
import math

def heavy_computation(n):
    # Simulated CPU-bound task
    return sum(math.isqrt(i) for i in range(n))

async def main():
    loop = asyncio.get_running_loop()
    
    # We use a ProcessPool to bypass the GIL
    with ProcessPoolExecutor() as pool:
        print("Starting CPU-bound task...")
        result = await loop.run_in_executor(pool, heavy_computation, 10_000_000)
        print(f"Task completed with result: {result}")

print("Hybrid concurrency pattern defined.")

## 5. Race Conditions & Deadlocks

### **Interview Question**: "How do you prevent two threads from modifying the same variable at the same time?"

**Answer**: Use a **Lock** (mutex). 
```python
lock = threading.Lock()
with lock:
    shared_resource += 1
```
Explain that while locks prevent race conditions, they can lead to **Deadlocks** if two threads are each waiting for a lock held by the other.

---